# 🌐 Ultimate LLM Chatbot
This notebook features **Qwen 7B**, real-time text streaming, **Speech-to-Text** (Microphone), **Text-to-Speech** (Audio replies), and **Web Search** (DuckDuckGo integration)!

## 1. Install Dependencies

In [ ]:
!pip install -qU gradio transformers accelerate bitsandbytes duckduckgo-search gTTS librosa soundfile
!apt-get install -y ffmpeg

## 2. Load the AI Models
Loading the Qwen 7B LLM in 4-bit precision, and the Whisper-tiny model for Voice Recognition.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer, pipeline
from threading import Thread
import gradio as gr
from duckduckgo_search import DDGS
from gtts import gTTS
import os

print("Loading LLM Tokenizer...")
llm_model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

print("Loading LLM in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16,
)

print("Loading Whisper STT model...")
whisper_stt = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en", device_map="auto")

print("All models loaded successfully!")

## 3. Define Tools & Chat Logic
This function handles audio transcription, web searching, text generation, and audio synthesis.

In [ ]:
def perform_web_search(query):
    print(f"Searching web for: {query}")
    try:
        results = DDGS().text(query, max_results=3)
        context = " ".join([r['body'] for r in results])
        return context
    except Exception as e:
        return "Search failed or no results."

def chat(text_input, audio_input, history, system_prompt, temperature, max_tokens, enable_search, enable_voice_reply):
    # 1. Handle Audio Input (Speech-to-Text)
    user_message = text_input
    if audio_input is not None:
        transcription = whisper_stt(audio_input)["text"]
        user_message = transcription.strip()
        
    if not user_message:
        yield history, None
        return

    # Update history with user message immediately
    history.append([user_message, ""])
    yield history, None

    # 2. Handle Web Search
    current_system_prompt = system_prompt
    if enable_search:
        search_context = perform_web_search(user_message)
        current_system_prompt += f"\n\n[Web Search Context (Use this to answer the user if relevant)]:\n{search_context}"

    # 3. Prepare Messages
    messages = [{"role": "system", "content": current_system_prompt}]
    for user_msg, bot_msg in history[:-1]:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": user_message})

    # 4. Generate text with Streaming
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    streamer = TextIteratorStreamer(tokenizer, timeout=10., skip_prompt=True, skip_special_tokens=True)
    generate_kwargs = dict(
        model_inputs,
        streamer=streamer,
        max_new_tokens=max_tokens,
        temperature=temperature,
        do_sample=True if temperature > 0 else False,
        top_p=0.9,
    )
    
    t = Thread(target=model.generate, kwargs=generate_kwargs)
    t.start()
    
    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        history[-1][1] = partial_text
        yield history, None

    # 5. Handle Audio Output (Text-to-Speech)
    audio_path = None
    if enable_voice_reply and partial_text.strip():
        try:
            tts = gTTS(text=partial_text, lang='en', slow=False)
            audio_path = "reply.mp3"
            tts.save(audio_path)
        except Exception as e:
            print("TTS Error:", e)
    
    yield history, audio_path

## 4. Build and Launch the Ultimate UI

In [ ]:
css = """
.gradio-container { max-width: 1000px !important; margin: auto !important; font-family: 'Inter', sans-serif; }
#header { text-align: center; padding: 25px; background: linear-gradient(135deg, #11998e, #38ef7d); color: white; border-radius: 16px; margin-bottom: 20px; box-shadow: 0 4px 15px rgba(0,0,0,0.1); }
#header h1 { font-size: 38px; margin: 0; font-weight: bold; }
#header p { font-size: 16px; opacity: 0.95; margin-top: 5px; }
.advanced-settings { border: 1px solid #e0e0e0; border-radius: 12px; padding: 15px; background-color: #f9f9f9; }
"""

with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:
    gr.HTML('''
    <div id="header">
        <h1>🌐 Ultimate AI Chatbot</h1>
        <p>Powered by Qwen 7B • Web Search • Voice Chat • Real-time Streaming</p>
    </div>
    ''')
    
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=500, bubble_full_width=False, avatar_images=(None, "https://cdn-icons-png.flaticon.com/512/4712/4712027.png"))
            
            with gr.Row():
                text_input = gr.Textbox(placeholder="Type your message here...", container=False, scale=7)
                audio_input = gr.Audio(sources=["microphone"], type="filepath", scale=1)
                submit_btn = gr.Button("Send", variant="primary", scale=1)
            
            audio_output = gr.Audio(label="Voice Reply", interactive=False, autoplay=True)
            clear_btn = gr.Button("🗑️ Clear Conversation", size="sm")
                
        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Superpowers & Settings", open=True, elem_classes="advanced-settings"):
                enable_search = gr.Checkbox(label="🌐 Enable Web Search", value=False)
                enable_voice_reply = gr.Checkbox(label="🗣️ Enable Voice Reply (TTS)", value=False)
                
                system_prompt = gr.Textbox(value="You are an intelligent assistant. Answer concisely and use the provided web search context if applicable.", label="System Prompt", lines=3)
                temperature = gr.Slider(minimum=0.0, maximum=2.0, value=0.7, step=0.1, label="Temperature")
                max_tokens = gr.Slider(minimum=1, maximum=4096, value=1024, step=64, label="Max Tokens")

    def clear_inputs():
        return "", None

    # Submit event binding
    submit_event = submit_btn.click(
        chat, 
        inputs=[text_input, audio_input, chatbot, system_prompt, temperature, max_tokens, enable_search, enable_voice_reply], 
        outputs=[chatbot, audio_output],
        queue=True
    ).then(clear_inputs, outputs=[text_input, audio_input])

    text_input.submit(
        chat, 
        inputs=[text_input, audio_input, chatbot, system_prompt, temperature, max_tokens, enable_search, enable_voice_reply], 
        outputs=[chatbot, audio_output],
        queue=True
    ).then(clear_inputs, outputs=[text_input, audio_input])

    clear_btn.click(lambda: ([], None), outputs=[chatbot, audio_output], queue=False)

demo.queue()
demo.launch(debug=True, share=True)
